Paquetes necesarios

In [1]:
import cv2
import time
import numpy as np

**TAREA**: Los ejemplos ilustrativos anteriores permiten saber el número de monedas presentes en la imagen ideal proporcionada. ¿Cómo determinar la cantidad de dinero presente en ella? A partir únicamente de las dimensiones, una sugerencia implicaría, en primer término, identificar de forma interactiva (por ejemplo haciendo clic en la imagen) una moneda de un valor fijo en la imagen (por ejemplo de 1€) para permitir establecer una relación entre píxeles y milímetros. Tras obtener esa información y las dimensiones en milímetros de las distintas monedas, es factible calcular la cantidad de dinero en la imagen ideal.

Una vez resuelto el reto con la imagen ideal proporcionada, diseña un demostrador en vivo que a partir de la entrada de vídeo de la cámara, determine el dinero presente en la escena. Si bien no hay restricciones sobre utilizar medidas geométricas o de color, o cualquier clasificador, no deben hacer uso de modelos de aprendizaje profundo. ¿Funciona correctamente? ¿Se observan problemas?
Posibles extras serían la presencia de objetos que no son monedas y/o haya solape entre las monedas.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo

Nota: Para establecer la correspondencia entre píxeles y milímetros, comentar que la moneda de un euro tiene un diámetro de 23.25 mm. la de 50 céntimos de 24.35, la de 20 céntimos de 22.25, etc. 



In [ ]:
vid = cv2.VideoCapture(0)

REFERENCE_OBJECT_MM = 27
pixel_size_buffer = []
pixel_size = 0 # mm/pixel

CALIBRATION_SS = 5
start_time = time.time()

amounts_readed = []

MONEY = {
    'cobre':   {8.125: 0.01, 9.375: 0.02, 10.625: 0.05},
    'dorado':  {9.875: 0.10, 11.125: 0.20, 12.125: 0.50},
    'bimetal': {11.625: 1.0, 12.875: 2.0},
    'billete': {(62, 120): 5, (67, 120): 10, (72, 133): 20, (77, 140): 50, (82, 147): 100, (82, 153): 200, (82, 160): 500}
}
TOLERANCE_MM = 0.8
TOLERANCE_BILL_MM = (2, 3)
H_LIMITE = 20

BORDER_PX = 20        # grosor de la franja de borde usada como muestra de fondo
DIST_MAX = 40.0       # distancia a/b que se mapea a 255 (ajustar mirando dist_u8)
MIN_SIGNAL = 8.0      # si ni el percentil 99.5 llega aquí, se considera que no hay objetos
CLOSE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
OPEN_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

def segment(frame):
    # 1. Desenfoque para reducir el ruido de croma
    blurred = cv2.GaussianBlur(frame, (5, 5), 0)

    # 2. Lab en uint8 -> float32 para poder restar con signo
    lab = cv2.cvtColor(blurred, cv2.COLOR_BGR2LAB).astype(np.float32)
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    # 3. Color del fondo: mediana de a y b en la franja de borde
    border = np.zeros(a.shape, dtype=bool)
    border[:BORDER_PX, :] = True
    border[-BORDER_PX:, :] = True
    border[:, :BORDER_PX] = True
    border[:, -BORDER_PX:] = True
    a_bg = np.median(a[border])
    b_bg = np.median(b[border])

    # 4. Distancia de crominancia al fondo
    dist = cv2.magnitude(a - a_bg, b - b_bg)

    # 5. Escalado con rango fijo a uint8
    dist_u8 = np.clip(dist * (255.0 / DIST_MAX), 0, 255).astype(np.uint8)

    # 6. Si no hay señal suficiente, devolvemos máscara vacía (evita segmentar ruido)
    if np.percentile(dist, 99.5) < MIN_SIGNAL:
        return np.zeros(a.shape, dtype=np.uint8), dist_u8

    # 7. Otsu con objetos en blanco (sin INV)
    _, binary = cv2.threshold(dist_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 8. Cierre para rellenar huecos (logo, parte plateada) y apertura para quitar motas
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, CLOSE_KERNEL)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, OPEN_KERNEL)

    return binary, dist_u8

def meassure_color(hsv, mask, dist_img, max_val):
    # Píxeles lejos del borde
    center = ((dist_img > 0.75 * max_val) & (mask > 0)).astype(np.uint8) * 255
    # Píxeles cerca del borde, pero sin tocarlo
    ring = ((dist_img > 0.08 * max_val) & (dist_img < 0.40 * max_val)
              & (mask > 0)).astype(np.uint8) * 255

    h_c, s_c, _, _ = cv2.mean(hsv, mask=center)
    h_r, s_r, _, _ = cv2.mean(hsv, mask=ring)
    return (h_c, s_c), (h_r, s_r)

def classify_group(centro, anillo, umbral_bimetal=12):
    (h_c, s_c), (h_r, s_r) = centro, anillo

    # Monedas con dos tonos (1€ y 2€) tienen un umbral de diferencia de tono más alto
    if abs(h_c - h_r) > umbral_bimetal or abs(s_c - s_r) > 40:
        return 'bimetal'

    # Monedas de un solo tono se clasifican en cobre o dorado según el valor medio del tono
    h = (h_c + h_r) / 2
    return 'cobre' if h < H_LIMITE else 'dorado'

def classify(radius, group):
    # Buscamos la moneda más cercana al radio medido para el grupo determinado
    key = min(MONEY[group], key=lambda k: abs(k - radius))
    if abs(key - radius) > TOLERANCE_MM:
        return None          # no encaja con ninguna moneda del grupo
    return MONEY[group][key]

def classify_bill(short, long, group):
    # Buscamos el bellite cuyo tamaño sea el más cercano al del contorno detectado
    key = min(MONEY[group], key=lambda k: abs(k[0] - short + k[1] - long))
    if abs(key[0] - short) > TOLERANCE_BILL_MM[0] and abs(key[1] - long) > TOLERANCE_BILL_MM[1]:
        return None          # no encaja con ningún billete
    return MONEY[group][key]

def create_countour_elements(binary_image, index):
    # Creamos una máscara para el contorno actual y calculamos el valor máximo de la transformada de distancia dentro de esa máscara
    mask = np.zeros(binary_image.shape, dtype=np.uint8)
    cv2.drawContours(mask, contours, index, 255, thickness=-1)
    distance_image = cv2.distanceTransform(mask, cv2.DIST_L2, 5)

    return mask, distance_image

def find_distance_from_center_px(contour, binary_image, index):
    # Eliminamos ruido de contornos pequeños
    if cv2.contourArea(contour) < 100:
        return None

    mask, distance_image = create_countour_elements(binary_image, index)

    _, distance_from_center, _, _ = cv2.minMaxLoc(distance_image, mask=mask)

    return distance_from_center

def calibrate(contours, binary_image):
    for index, contour in enumerate(contours):
        distance = find_distance_from_center_px(contour, binary_image, index)

        if distance:
            # Calculamos el tamaño de pixel a partir del radio de referencia y lo añadimos al buffer
            pixel_size_buffer.append(REFERENCE_OBJECT_MM / distance)

def get_bill_shape(contour):
    # Obtenemos el ancho y largo del contorno del billete
    _, (w, h), _ = cv2.minAreaRect(contour)

    # Ordenamos w y h para obtener siempre el mismo orden, sin importar el ángulo
    short_px, long_px = sorted((w, h))

    # Conversión a mm con la escala de la calibración
    short_mm = short_px * pixel_size
    long_mm = long_px * pixel_size

    # Relación de aspecto (con protección frente a división por cero)
    aspect = long_px / short_px if short_px > 0 else 0

    return short_mm, long_mm, aspect

def get_amount(contours, binary_image, hsv):
    total_amount = 0
    for index, contour in enumerate(contours):
        distance = find_distance_from_center_px(contour, binary_image, index)

        if distance is None or distance <= 0:
            continue

        mask, distance_image = create_countour_elements(binary_image, index)

        # Calculamos el radio en mm a partir del valor máximo de la transformada de distancia y el tamaño de pixel
        radius = distance * pixel_size

        if radius > 20:
            short_mm, long_mm, aspect = get_bill_shape(contour)

            # Utilizamos la relación de aspectos para evitar leer elementos que no sean similares a billetes
            if aspect < 1.7 or aspect > 2.05:
                continue

            value = classify_bill(short_mm, long_mm, 'billete')

            if value is not None:
                total_amount += value
                print(f'Billete: {value}€ - Tamaño: {short_mm:.2f}x{long_mm:.2f}mm')
            else:
                print(f'Billete desconocido - Tamaño: {short_mm:.2f}x{long_mm:.2f}mm')
        else:
            # Establecemos el grupo al que pertenece la moneda en base a su color
            measurements = meassure_color(hsv, mask, distance_image, distance)
            group = classify_group(*measurements)

            if group is not None:
                # Clasificamos la moneda y sumamos su valor al total de la mesa
                value = classify(radius, group)
                if value is not None:
                    total_amount += value
                    print(f'Moneda: {value}€ - Radio: {radius:.2f}mm - Grupo: {group}')
                else:
                    print(f'Moneda desconocida - Radio: {radius:.2f}mm - Grupo: {group}')

    # Añadimos el total de la mesa a la lista de lecturas
    amounts_readed.append(total_amount)
    # Se muestra el valor de la mesa como la mediana de las lecturas obtenidas
    print(f'Valor en la mesa: {np.median(amounts_readed)}')

while(True):
    # Leemos la entrada de la cámara fotograma a fotograma
    ret, frame = vid.read()

    if ret:
        binarized_image, dist_u8 = segment(frame)
        contours, _ = cv2.findContours(binarized_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Dibujamos los contornos sobre la imagen original para mostrar al usuario
        rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        cv2.drawContours(rgb_image, contours, -1, (0,255,0), 3)
        bgr_frame = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)

        # Calibramos el fondo durante los primeros segundos
        if time.time() - start_time < CALIBRATION_SS:
            # Leemos el radio del contorno de referencia y establecemos un tamaño de pixel
            calibrate(contours, binarized_image)

            # Calculamos el tamaño de pixel a partir de la mediana de las lecturas previas
            pixel_size = np.median(pixel_size_buffer)

            # Mostramos los fotogramas para aportar feedback visual al usuario
            cv2.imshow('Fotograma', bgr_frame)

            # Detenemos pulsando ESC
            if cv2.waitKey(20) == 27:
                break

            continue
        # Nos aseguramos de que el tamaño de pixel no sea cero antes de continuar con la medición
        if pixel_size == 0:
            print("Error: El tamaño de pixel es cero. La calibración puede haber fallado.")
            break

        get_amount(contours, binarized_image, hsv)
        # Mostramos los fotogramas para aportar feedback visual al usuario
        cv2.imshow('Fotograma', bgr_frame)

    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

# Liberamos el objeto de captura y destruimos las ventanas
vid.release()
cv2.destroyAllWindows()

Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en la mesa: 0.0
Valor en l

**TAREA**: La tarea consiste en extraer características (geométricas y/o visuales) de las tres imágenes completas de partida, y *aprender* patrones que permitan identificar las partículas en nuevas imágenes. Para ello se proporciona como imagen de test *MPs_test.jpg* y sus correpondientes anotaciones *MPs_test_bbs.csv* con la que deben obtener las métricas para su propuesta de clasificación de microplásticos, además de la matriz de confusión. La matriz de confusión permitirá mostrar para cada clase el número de muestras que se clasifican correctamente de dicha clase, y el número de muestras que se clasifican incorrectamente como perteneciente a una de las otras dos clases. De nuevo, no deben hacer uso de modelos de aprendizaje profundo ni para segmentar, ni para clasificar.

En el trabajo [SMACC: A System for Microplastics Automatic Counting and Classification](https://doi.org/10.1109/ACCESS.2020.2970498), las características geométricas utilizadas fueron:

- Área en píxeles
- Perímetro en píxeles
- Compacidad (relación entre el cuadrado del perímetro y el área de la partícula)
- Relación del área de la partícula con la del contenedor
- Relación del ancho y el alto del contenedor
- Relación entre los ejes de la elipse ajustada
- Definido el centroide, relación entre las distancias menor y mayor al contorno

Si no se quedan satisfechos con la segmentación obtenida, es el mundo real, también en el README comento técnicas recientes de segmentación, que podrían despertar su curiosidad.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo